# 04 Generation & Evaluation

`src.generation`?? RAG ??? ??? `src.evaluation`?? ?? ??? ?????.



In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

print(f"Project root: {ROOT}")



In [ ]:
from src.common.config import get_settings
from src.retrieval.factory import build_retriever
from src.generation.llm import OllamaGenerator
from src.generation.rag_pipeline import RAGPipeline

settings = get_settings()

retriever = build_retriever(mode="hybrid", k=5)
generator = OllamaGenerator(model=settings.ollama_model, base_url=settings.ollama_base_url)
rag = RAGPipeline(retriever, generator)



In [ ]:
query = "KIKO ??? ????? ??? ??? ??????"
result = rag.answer(query)
print(result["answer"])
print("retrieved_ids:", result["retrieved_ids"])



In [ ]:
from src.evaluation.pipeline import run_evaluation

df = run_evaluation(
    eval_csv_path=settings.default_eval_csv_path,
    chunk_json_path=settings.default_chunk_json_path,
    output_csv_path=settings.eval_data_dir / "eval_result.csv",
    retrieval_mode="hybrid",
    ollama_model=settings.ollama_model,
    ollama_base_url=settings.ollama_base_url,
    dense_provider="clova",
    dense_model_name="bge-m3",
    dense_collection_name="docs_clova",
    dense_persist_directory=str(settings.chroma_clova_dir),
    k=5,
)

print(df[["hit", "recall", "bert_score_f1"]].mean(numeric_only=True))

